# MambaTree in the DAM

Lab notebook for replacing the DAM's scan with a tree-topology scan (MambaTree).

**How this is organised.** Model code lives in `.py` files so `main.py` / `train.py` and the server runs pick it up; this notebook imports them with autoreload, explains each change, and runs the checks. Edit the `.py`, re-run the cells.

| # | Date | Change | Files | Checks |
|---|---|---|---|---|
| 0 | | Baseline: bidirectional raster scan (current DAM) | `components/dam.py` | §2, §3 |
| 1 | | Option A: MST tree scan replaces `_bidir_scan` (TIS + SDS kept, GrootV LayerNorm on state) | `components/dam_tree.py` | §2, §3, §4 |
| 2 | | | | |

## 0. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# the MOCID folder, whether the notebook runs from MOCID/ or MOCID/notebooks/
MOCID_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(MOCID_DIR))

import torch

from tools.scan_ref import install_cpu_scan

# no CUDA (Mac): swap in the pure-PyTorch scan before dam.py imports the kernel
print("cpu reference scan:", install_cpu_scan())

from components.dam import DAMBlock, DisplacementNet, TIDS

torch.manual_seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CH = [128, 256, 512]  # base_channels=16 -> 1/8, 1/16, 1/32 scales
D_STATE = 32  # MOCID default

## 1. Where the scan sits

```
DisplacementNet           per scale k, per reference frame r
 └─ DAMBlock(F_T, F_R)    norm -> in_x -> dwconv -> SiLU  (xt, xr)   |   in_z -> SiLU (gate z)
     └─ TIDS(xt, xr)
         ├─ SDS           3D-CDC([F_R, F_T]) -> dt, B, C        (scan params, full res)
         ├─ TIS           avgpool + interleave ref/target -> X_W (along W), X_H (along H)
         └─ _bidir_scan   flatten to L = H*W, scan forward + reverse, add   <-- swap point
     └─ out(F_R + mid(TIDS * z))   mid = 0 at init -> returns F_R (handover)
```

The raster scan flattens each 2D map into a 1D sequence, so spatial neighbours across rows end up W tokens apart. A tree scan builds a minimum spanning tree over the 4-connected pixel grid (edge weight = feature dissimilarity) and propagates state along the tree, so it runs on the 2D map directly and the flatten/transpose goes away.

Two ways to slot it in:
- **A. Replace `_bidir_scan` only.** Keep SDS (dt, B, C) and the TIS interleave; scan X_W and X_H with the tree instead of the raster order. Smallest change, isolates the effect of the scan.
- **B. Replace TIS + scan.** Build one tree over the stacked (ref, target) pair with spatial and temporal edges, dropping the interleave. Bigger change, closer to a new method.

Trace the shapes going into the swap point:

In [ ]:
def trace_scan(tids, H=8, W=8, B=1):
    # wrap _bidir_scan to print what it receives and returns
    orig = tids._bidir_scan

    def wrapped(seq, dt, Bp, Cp, A):
        out = orig(seq, dt, Bp, Cp, A)
        print(f"  seq {tuple(seq.shape)}  dt {tuple(dt.shape)}  B/C {tuple(Bp.shape)}  "
              f"A {tuple(A.shape)} -> {tuple(out.shape)}")
        return out

    tids._bidir_scan = wrapped
    x = torch.randn(B, tids.d_inner, H, W)
    y = tids(x, torch.randn_like(x))
    tids._bidir_scan = orig
    print(f"TIDS {tuple(x.shape)} x2 -> {tuple(y.shape)}")


trace_scan(TIDS(CH[0], d_state=D_STATE))

## 2. Scan contract

Any replacement TIDS must: take `xt, xr (B, d_inner, H, W)`, return the same shape, stay finite, and pass gradients to its own parameters (`A_log`, `D`, SDS).

In [ ]:
def check_tids(tids, H=8, W=8, B=2):
    # -> dict of pass/fail for one TIDS variant
    xt = torch.randn(B, tids.d_inner, H, W, device=DEVICE, requires_grad=True)
    xr = torch.randn_like(xt, requires_grad=True)
    y = tids(xt, xr)
    y.square().mean().backward()

    no_grad = [n for n, p in tids.named_parameters() if p.grad is None or p.grad.abs().sum() == 0]
    return {
        "shape": tuple(y.shape) == tuple(xt.shape),
        "finite": bool(torch.isfinite(y).all()),
        "grad_in": xt.grad is not None and xr.grad is not None,
        "params_no_grad": no_grad or "none",
    }

## 3. Block checks

Mirrors `tools/check_dam.py` without needing a checkpoint:
- **handover**: at init the block returns `F_R` exactly (so stage 2 starts from stage 1)
- **params**: DAM total over the three scales vs the paper's 3.60 M

In [ ]:
def check_block(make_tids=None, C=CH[0], H=8, W=8, B=2):
    # make_tids(C) -> TIDS variant; None keeps the current one
    block = DAMBlock(C, d_state=D_STATE).to(DEVICE)
    if make_tids is not None:
        block.tids = make_tids(C).to(DEVICE)
    F_T = torch.randn(B, C, H, W, device=DEVICE)
    F_R = torch.randn_like(F_T)
    with torch.no_grad():
        err = (block(F_T, F_R) - F_R).abs().max().item()
    return {"handover_max_err": err, "handover": err < 1e-5}


def dam_params(make_tids=None):
    # -> DAM parameters in millions across all scales
    net = DisplacementNet(CH, d_state=D_STATE)
    if make_tids is not None:
        for blk, c in zip(net.blocks, CH):
            blk.tids = make_tids(c)
    return sum(p.numel() for p in net.parameters()) / 1e6

## 4. Variants

`TreeTIDS` (`components/dam_tree.py`) subclasses `TIDS`: same SDS and interleave, `_bidir_scan` replaced by `_tree_scan`.

- **Tree:** MST over the 4-connected grid of the interleaved map, edge cost `exp(-cosine similarity)` (GrootV's choice), so similar neighbours are joined first.
- **Scan:** leaf -> root then root -> leaf, so every pixel receives every other pixel weighted by the product of decays along the tree path. One pass replaces forward + reverse.
- **State:** each (channel, state) pair is one channel of the tree filter, so cost grows with `d_inner * d_state`. GrootV uses `d_state=1`; both are listed below.
- **Norm:** LayerNorm on the aggregated state before the read-out (GrootV's `h_norm`), since the tree sums over all pixels.

On a Mac this runs the pure-PyTorch reference (slow, keep maps small). On the server it uses GrootV's CUDA ops once built (§6).

In [ ]:
from components.dam_tree import TreeTIDS, HAS_TREE_CUDA

VARIANTS = {
    "bidir (baseline)": lambda C: TIDS(C, d_state=D_STATE),
    "tree": lambda C: TreeTIDS(C, d_state=D_STATE),
    "tree, d_state=1 (GrootV)": lambda C: TreeTIDS(C, d_state=1),
}
print("tree cuda ops:", HAS_TREE_CUDA)

rows = []
for name, make in VARIANTS.items():
    r = {"variant": name}
    r.update(check_tids(make(CH[0]).to(DEVICE)))
    r.update(check_block(make))
    r["DAM params (M)"] = round(dam_params(make), 3)
    rows.append(r)

try:
    import pandas as pd
    display(pd.DataFrame(rows).set_index("variant"))
except ImportError:
    for r in rows:
        print(r)

## 6. CUDA backend parity (server only)

Build GrootV's ops once in the server venv:

```bash
git clone https://github.com/EasonXiao-888/MambaTree.git
cd MambaTree/GrootV/third-party/TreeScan && pip install -v -e .
```

Then check the CUDA path matches the reference on the same inputs (forward and gradients). Ties in edge cost can give a different but equally valid tree, so small mismatches on random inputs are worth a second look rather than an immediate alarm.

In [ ]:
if torch.cuda.is_available() and HAS_TREE_CUDA:
    torch.manual_seed(0)
    tids = TreeTIDS(CH[0], d_state=4).cuda()
    xt = torch.randn(2, CH[0], 8, 8, device="cuda", requires_grad=True)
    xr = torch.randn_like(xt, requires_grad=True)

    res = {}
    for be in ["ref", "cuda"]:
        tids.backend = be
        tids.zero_grad()
        xt.grad = None
        y = tids(xt, xr)
        y.square().mean().backward()
        res[be] = (y.detach(), xt.grad.clone(), tids.A_log.grad.clone())

    for name, a, b in zip(["out", "grad xt", "grad A_log"], res["ref"], res["cuda"]):
        print(f"{name:10s} max abs diff {(a - b).abs().max().item():.2e}")
else:
    print("skipped: needs CUDA and the tree_scan extension")

## 5. Notes and results

Log decisions and training results per variant here (run name, stage, AP50, F1, params, inference time).